# TCN Baseline: HDFS_v1 (BlockId Trace -> Anomaly / Normal)

Trains a TCN binary classifier on HDFS_v1 preprocessed BlockId traces from:
- `Dataset/HDFS_v1/preprocessed/HDFS.npz` (`x_data`, `y_data`)

Prints per-epoch `train_acc`/`val_acc` and a final `test_acc`.
No extra preprocessing is needed for HDFS_v1 because `HDFS.npz` already contains the traces + labels.


In [17]:
import json
import subprocess
import sys
from datetime import datetime
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(8):
        if (p / 'Dataset').exists() and (p / 'Baselines').exists():
            return p
        p = p.parent
    raise RuntimeError('Could not find repo root (expected Dataset/ and Baselines/)')

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR)
TCN_DIR = REPO_ROOT / 'Baselines' / 'TCN'

print('NOTEBOOK_DIR:', NOTEBOOK_DIR)
print('REPO_ROOT   :', REPO_ROOT)
print('TCN_DIR     :', TCN_DIR)


NOTEBOOK_DIR: C:\Users\nikhi\OneDrive\Attachments\Desktop\Git_Repos\Log_Anomly_Detection\Baselines\TCN
REPO_ROOT   : C:\Users\nikhi\OneDrive\Attachments\Desktop\Git_Repos\Log_Anomly_Detection
TCN_DIR     : C:\Users\nikhi\OneDrive\Attachments\Desktop\Git_Repos\Log_Anomly_Detection\Baselines\TCN


In [18]:
# Pick a Python that has torch + numpy installed.
# Common failure mode: your Jupyter kernel python is different from the python where torch is installed.
DEFAULT_PYTHON = r'C:\Users\nikhi\AppData\Local\Programs\Python\Python311\python.exe'

def can_import_torch(python_exe: str) -> bool:
    try:
        r = subprocess.run(
            [python_exe, '-c', 'import torch, numpy'],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
        return r.returncode == 0
    except Exception:
        return False

candidates = []
if sys.executable:
    candidates.append(sys.executable)
candidates.append(DEFAULT_PYTHON)

PYTHON = None
for c in candidates:
    if c and Path(c).exists() and can_import_torch(c):
        PYTHON = c
        break

if PYTHON is None:
    raise RuntimeError(
        'No usable Python found that can import torch + numpy. '
        'Install PyTorch into your Jupyter kernel env, or edit DEFAULT_PYTHON to the correct python.exe.'
    )

print('Selected PYTHON:', PYTHON)


Selected PYTHON: c:\Users\nikhi\AppData\Local\Programs\Python\Python311\python.exe


## Hyperparameters
Edit these values and rerun the training cell.


In [19]:
HDFS_NPZ = REPO_ROOT / 'Dataset' / 'HDFS_v1' / 'preprocessed' / 'HDFS.npz'

SEQ_LEN = 64
MAX_TRACES = 50000
SEED = 42
VAL_FRAC = 0.1
TEST_FRAC = 0.1

EPOCHS = 30
BATCH_SIZE = 128
LR = 1e-3

EMBEDDING_DIM = 32
CHANNELS = '64,64,64'
KERNEL_SIZE = 3
DROPOUT = 0.2

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
TRAIN_LOG = TCN_DIR / f'train_hdfs_v1_{RUN_ID}.log'
RUN_SUMMARY = TCN_DIR / f'run_hdfs_v1_{RUN_ID}.summary.json'

print('HDFS_NPZ   :', HDFS_NPZ)
print('exists?    :', HDFS_NPZ.exists())
print('TRAIN_LOG  :', TRAIN_LOG)
print('RUN_SUMMARY:', RUN_SUMMARY)


HDFS_NPZ   : C:\Users\nikhi\OneDrive\Attachments\Desktop\Git_Repos\Log_Anomly_Detection\Dataset\HDFS_v1\preprocessed\HDFS.npz
exists?    : True
TRAIN_LOG  : C:\Users\nikhi\OneDrive\Attachments\Desktop\Git_Repos\Log_Anomly_Detection\Baselines\TCN\train_hdfs_v1_20260315_193811.log
RUN_SUMMARY: C:\Users\nikhi\OneDrive\Attachments\Desktop\Git_Repos\Log_Anomly_Detection\Baselines\TCN\run_hdfs_v1_20260315_193811.summary.json


## Train
Runs `train_tcn_hdfs_v1.py` and streams output to the notebook and a log file.


In [20]:
train_cmd = [
    PYTHON,
    str(TCN_DIR / 'train_tcn_hdfs_v1.py'),
    '--hdfs_npz', str(HDFS_NPZ),
    '--seq_len', str(SEQ_LEN),
    '--max_traces', str(MAX_TRACES),
    '--seed', str(SEED),
    '--val_frac', str(VAL_FRAC),
    '--test_frac', str(TEST_FRAC),
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--lr', str(LR),
    '--embedding_dim', str(EMBEDDING_DIM),
    '--channels', str(CHANNELS),
    '--kernel_size', str(KERNEL_SIZE),
    '--dropout', str(DROPOUT),
]

print(' '.join(train_cmd))
with open(TRAIN_LOG, 'w', encoding='utf-8') as f:
    p = subprocess.Popen(train_cmd, cwd=str(TCN_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
        f.write(line)
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f'Training failed with exit code {rc}. See {TRAIN_LOG}')

ckpt = TCN_DIR / 'artifacts' / 'tcn_hdfs_v1_best.pt'
print('Training log:', TRAIN_LOG)
print('Checkpoint  :', ckpt)

summary = {
    'run_id': RUN_ID,
    'hdfs_npz': str(HDFS_NPZ),
    'train_log': str(TRAIN_LOG),
    'checkpoint': str(ckpt),
    'params': {
        'seq_len': SEQ_LEN,
        'max_traces': MAX_TRACES,
        'seed': SEED,
        'val_frac': VAL_FRAC,
        'test_frac': TEST_FRAC,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'embedding_dim': EMBEDDING_DIM,
        'channels': CHANNELS,
        'kernel_size': KERNEL_SIZE,
        'dropout': DROPOUT,
    },
    'python': PYTHON,
    'cwd': str(TCN_DIR),
    'timestamp': datetime.now().isoformat(timespec='seconds'),
}
with open(RUN_SUMMARY, 'w', encoding='utf-8') as sf:
    json.dump(summary, sf, indent=2)
print('Run summary:', RUN_SUMMARY)


c:\Users\nikhi\AppData\Local\Programs\Python\Python311\python.exe C:\Users\nikhi\OneDrive\Attachments\Desktop\Git_Repos\Log_Anomly_Detection\Baselines\TCN\train_tcn_hdfs_v1.py --hdfs_npz C:\Users\nikhi\OneDrive\Attachments\Desktop\Git_Repos\Log_Anomly_Detection\Dataset\HDFS_v1\preprocessed\HDFS.npz --seq_len 64 --max_traces 50000 --seed 42 --val_frac 0.1 --test_frac 0.1 --epochs 30 --batch_size 128 --lr 0.001 --embedding_dim 32 --channels 64,64,64 --kernel_size 3 --dropout 0.2
c:\Users\nikhi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
epoch=1 train_acc=0.996150 train_p=0.951510 train_r=0.911481 train_f1=0.931065 val_acc=0.994400 val_p=0.916084 val_r=0.891156 val_f1=0.903448
epoch=2 train_acc=0.998800 train_p=0.976460 train_r=0.981595 train_f1=0.979021 val_acc=0.997800 val_p=0.953333 va